# AIC354 - Machine Learning Fundamentals
## Lab 05: Naïve Bayes Classifier
### Graded Lab Tasks Solution Notebook
---
**Contents:**
1. **Graded Lab Task 1:** Wine Classification using Naive Bayes (GaussianNB vs MultinomialNB)
2. **Graded Lab Task 2:** Loan Default Prediction using Naive Bayes with Exploratory Data Analysis (EDA)


---
# Lab Task 1: Wine Classification (Sklearn Dataset)
**Task Statement:**
> Use wine dataset from `sklearn.datasets` to classify wines into 3 categories. Load the dataset and split it into test and train. After that train the model using Gaussian and Multinomial classifier and evaluate which performs better. Use the trained model to perform some predictions on test data.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


### 1.1 Load and Explore Wine Dataset


In [ ]:
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
target_names = wine.target_names

wine_df = pd.DataFrame(X, columns=feature_names)
wine_df['target'] = y
wine_df['class_name'] = [target_names[i] for i in y]

print(f'Dataset Shape: {wine_df.shape}')
print(f'Target Classes: {list(target_names)}')
print('\nClass counts:')
print(wine_df['class_name'].value_counts())
wine_df.head()


### 1.2 Split Dataset into Train and Test Sets


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f'Training set samples: {X_train.shape[0]}')
print(f'Testing set samples:  {X_test.shape[0]}')


### 1.3 Train GaussianNB and MultinomialNB Models


In [ ]:
# Train Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_gnb = gnb.predict(X_test)
acc_gnb = accuracy_score(y_test, y_pred_gnb)

# Train Multinomial Naive Bayes
mnb = MultinomialNB()
mnb.fit(X_train, y_train)
y_pred_mnb = mnb.predict(X_test)
acc_mnb = accuracy_score(y_test, y_pred_mnb)

print(f'GaussianNB Accuracy:    {acc_gnb * 100:.2f}%')
print(f'MultinomialNB Accuracy: {acc_mnb * 100:.2f}%')


### 1.4 Evaluate and Compare Performance


In [ ]:
print('=== GaussianNB Classification Report ===')
print(classification_report(y_test, y_pred_gnb, target_names=target_names))

print('=== MultinomialNB Classification Report ===')
print(classification_report(y_test, y_pred_mnb, target_names=target_names))

# 5-fold Cross-Validation Comparison
cv_gnb = cross_val_score(GaussianNB(), X, y, cv=5).mean()
cv_mnb = cross_val_score(MultinomialNB(), X, y, cv=5).mean()
print(f'GaussianNB 5-Fold Cross-Validation:    {cv_gnb * 100:.2f}%')
print(f'MultinomialNB 5-Fold Cross-Validation: {cv_mnb * 100:.2f}%')


In [ ]:
# Confusion Matrix Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
disp_gnb = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_gnb), display_labels=target_names)
disp_gnb.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'GaussianNB (Acc: {acc_gnb*100:.1f}%)', fontweight='bold')

disp_mnb = ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_mnb), display_labels=target_names)
disp_mnb.plot(ax=axes[1], cmap='Oranges', colorbar=False)
axes[1].set_title(f'MultinomialNB (Acc: {acc_mnb*100:.1f}%)', fontweight='bold')
plt.tight_layout()
plt.show()


#### Why Gaussian Naive Bayes Performs Better:
- **GaussianNB** assumes features are continuous and normally distributed. Chemical and physical measurements in the wine dataset (alcohol, phenols, hue, flavanoids) are continuous real numbers, which matches the Gaussian assumption.
- **MultinomialNB** assumes features represent discrete count/frequency occurrences (e.g., word frequencies in text classification). Continuous float variables violate this assumption, leading to lower performance.


### 1.5 Predictions on Test Data


In [ ]:
sample_indices = [0, 5, 10, 15, 20]
sample_X = X_test[sample_indices]
sample_y = y_test[sample_indices]
predictions = gnb.predict(sample_X)
probabilities = gnb.predict_proba(sample_X)

results_df = pd.DataFrame({
    'Sample Index': sample_indices,
    'Actual Class': [target_names[i] for i in sample_y],
    'Predicted Class': [target_names[i] for i in predictions],
    'Confidence': [f'{np.max(p)*100:.2f}%' for p in probabilities],
    'Result': ['CORRECT' if a == p else 'INCORRECT' for a, p in zip(sample_y, predictions)]
})
results_df


---
# Lab Task 2: Loan Default Prediction (Real Dataset)
**Task Statement:**
> Train the Naive Bayes Classifier on the real dataset i.e. loan dataset. Perform data exploration to understand more about the dataset. Then develop a model to predict the customers who have not fully paid the loan.


### 2.1 Load Dataset and Exploratory Data Analysis (EDA)


In [ ]:
loan_df = pd.read_csv('loan_data.csv')
print(f'Shape: {loan_df.shape}')
loan_df.info()


In [ ]:
# Check missing values
print('Missing values:')
print(loan_df.isnull().sum())

# Summary statistics
loan_df.describe().round(2)


In [ ]:
# Target variable breakdown
target_counts = loan_df['not.fully.paid'].value_counts()
print('Class distribution:')
print(target_counts)
print(f'Fully Paid (0):     {target_counts[0]} ({target_counts[0]/len(loan_df)*100:.2f}%)')
print(f'Not Fully Paid (1): {target_counts[1]} ({target_counts[1]/len(loan_df)*100:.2f}%)')

# Repayment status by loan purpose
print('\nDefault rate by purpose:')
loan_df.groupby('purpose')['not.fully.paid'].agg(['count', 'sum', 'mean']).rename(
    columns={'count': 'Total', 'sum': 'Defaults', 'mean': 'Default_Rate'}
).sort_values(by='Default_Rate', ascending=False)


In [ ]:
# EDA Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='not.fully.paid', data=loan_df, ax=axes[0], palette=['#2ecc71', '#e74c3c'], hue='not.fully.paid', legend=False)
axes[0].set_title('Distribution of Loan Repayment Status', fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Fully Paid (84%)', 'Not Fully Paid (16%)'])

sns.countplot(y='purpose', hue='not.fully.paid', data=loan_df, ax=axes[1], palette=['#3498db', '#e67e22'])
axes[1].set_title('Loan Purpose by Repayment Status', fontweight='bold')
axes[1].legend(['Fully Paid', 'Not Fully Paid'])
plt.tight_layout()
plt.show()


In [ ]:
# FICO score distribution
plt.figure(figsize=(10, 5))
sns.histplot(data=loan_df, x='fico', hue='not.fully.paid', bins=30, kde=True, palette=['#2ecc71', '#e74c3c'])
plt.title('FICO Score Distribution by Repayment Status', fontweight='bold')
plt.xlabel('FICO Score')
plt.legend(['Not Fully Paid (1)', 'Fully Paid (0)'])
plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix
plt.figure(figsize=(11, 8))
corr = loan_df.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Numerical Attributes', fontweight='bold')
plt.tight_layout()
plt.show()


### 2.2 Data Preprocessing and Feature Encoding


In [ ]:
# One-Hot Encode categorical purpose variable
encoded_loan = pd.get_dummies(loan_df, columns=['purpose'], drop_first=True)

X_loan = encoded_loan.drop('not.fully.paid', axis=1)
y_loan = encoded_loan['not.fully.paid']

# Train/test split with stratification
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_loan, y_loan, test_size=0.3, random_state=42, stratify=y_loan
)

print(f'Training set shape: {X_train_l.shape}')
print(f'Testing set shape:  {X_test_l.shape}')


### 2.3 Train Naive Bayes Classifier


In [ ]:
gnb_loan = GaussianNB()
gnb_loan.fit(X_train_l, y_train_l)
print('Gaussian Naive Bayes model trained successfully!')


### 2.4 Model Evaluation


In [ ]:
y_pred_l = gnb_loan.predict(X_test_l)
y_prob_l = gnb_loan.predict_proba(X_test_l)[:, 1]

acc_l = accuracy_score(y_test_l, y_pred_l)
auc_l = roc_auc_score(y_test_l, y_prob_l)

print(f'Accuracy:  {acc_l * 100:.2f}%')
print(f'ROC-AUC:   {auc_l:.4f}')
print('\nClassification Report:')
print(classification_report(y_test_l, y_pred_l, target_names=['Fully Paid (0)', 'Not Fully Paid (1)']))


In [ ]:
# Confusion Matrix Plot
fig, ax = plt.subplots(figsize=(6, 5))
cm_l = confusion_matrix(y_test_l, y_pred_l)
disp_l = ConfusionMatrixDisplay(confusion_matrix=cm_l, display_labels=['Fully Paid', 'Not Fully Paid'])
disp_l.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Loan Default Confusion Matrix (Accuracy: {acc_l*100:.2f}%)', fontweight='bold')
plt.tight_layout()
plt.show()


### 2.5 Predicting Default for Future/Sample Customers


In [ ]:
sample_indices = [12, 72, 85, 108, 128]
sample_customers = X_test_l.iloc[sample_indices]
actual_classes = y_test_l.iloc[sample_indices].values
pred_classes = gnb_loan.predict(sample_customers)
pred_probs = gnb_loan.predict_proba(sample_customers)

customer_summary = pd.DataFrame({
    'Applicant': [f'Customer {idx}' for idx in sample_indices],
    'FICO Score': sample_customers['fico'].values,
    'Interest Rate': [f'{r*100:.2f}%' for r in sample_customers['int.rate'].values],
    'DTI Ratio': sample_customers['dti'].values,
    'Actual': ['Not Fully Paid' if a == 1 else 'Fully Paid' for a in actual_classes],
    'Predicted': ['RISK: Not Fully Paid' if p == 1 else 'SAFE: Fully Paid' for p in pred_classes],
    'Default Probability': [f'{p[1]*100:.2f}%' for p in pred_probs]
})
customer_summary
